In [5]:
path = r"D:\Akseli\Code\video_features\output\s3d\2025-05-28_120_Freddy-cam-1_cropped_s3d.npy"

# Load I3D features
data = np.load(path)

import numpy as np
import scipy.io as sio

# Load the trial data to get original event boundaries
trial_path = r"C:\Users\Julius\Desktop\Trial_data.mat"
mat_data = sio.loadmat(trial_path, squeeze_me=True, struct_as_record=False)
AllTrials = mat_data['AllTrials']

# Find the trial that matches this video (trial 33 based on filename)
trial_num = 120  # Extract from filename: 2025-05-28_033_Freddy-cam-1_cropped
matching_trial = None
for trial in AllTrials:
    if trial.trial_num == trial_num:
        matching_trial = trial
        break

if matching_trial is None:
    raise ValueError(f"Trial {trial_num} not found in trial data")

# Get original event duration in frames

stick_in_out_disp0 = int(trial.info.stick_in_out_disp[0]) - 1  # Convert to 0-based index
if stick_in_out_disp0 - 200 > 0:
    start_frame = stick_in_out_disp0 - 200
else:
    start_frame = 0
end_frame = trial.beakTip.xyz.shape[0] # since we access via shape, no need to do -1


# print end_frame - start_frame
x = end_frame - start_frame
print(f"Trial {trial_num} - Start frame: {start_frame}, End frame: {end_frame}, Duration in frames: {x}")


# Load I3D features
data = np.load(path)
print(f"Loaded I3D features shape: {data.shape}")

Trial 120 - Start frame: 1246, End frame: 2356, Duration in frames: 1110
Loaded I3D features shape: (1110, 1024)


In [ ]:
import os
import cv2
import numpy as np
import scipy.io as sio





video_folder = r"D:\Alice\VidData\20250528_01_Freddy"
trial_path = r"C:\Users\Julius\Desktop\Trial_data.mat"
output_folder = r".\sample"

text_file = r".\sample\sample_video_paths.txt"

os.makedirs(output_folder, exist_ok=True)

stack_size = 50  # for padding
startPreDispOut = 200 # Interested in video frames 200 frames before disp out

mat_data = sio.loadmat(trial_path, squeeze_me=True, struct_as_record=False)
AllTrials = mat_data['AllTrials']

# delete and recreate the text file
if os.path.exists(text_file):
    os.remove(text_file)

for trial in AllTrials:
    trial_num = trial.trial_num


    if trial.info.stick_in_out_disp is None:
        print(f"Skipping trial {trial_num} due to missing stick_in_out_disp")
        continue



    # files in video_folder look like 2025-05-28_001_Freddy-cam-1, where 001 is the trial number
    trial_num_str = f"{trial_num:03d}"
    matching_files = [f for f in os.listdir(video_folder) if f.endswith("cam-1.mp4") and f.split('_')[1] == trial_num_str]
    if matching_files:
        matching_file = matching_files[0]
    else:
        raise FileNotFoundError(f"No matching video file found for trial number {trial_num_str} in {video_folder}")



    stick_in_out_disp0 = int(trial.info.stick_in_out_disp[0]) - 1  # Convert to 0-based index
    if stick_in_out_disp0 - startPreDispOut > 0:
        start_frame = stick_in_out_disp0 - startPreDispOut
    else:
        start_frame = 0
    end_frame = trial.beakTip.xyz.shape[0] # since we access via shape, no need to do -1


    
    video_path = os.path.join(video_folder, matching_file)
    cap = cv2.VideoCapture(video_path)

    # print num 

    input_fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Total frames in video: {total_frames}")


    cropped_filename = f"{os.path.splitext(matching_file)[0]}_cropped.mp4"
    cropped_path = os.path.join(output_folder, cropped_filename)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    size = (224, 224)
    out = cv2.VideoWriter(cropped_path, fourcc, input_fps, size)

    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    for frame_idx in range(start_frame, end_frame):
        ret, frame = cap.read()
        if not ret:
            break
        out.write(cv2.resize(frame, size))


    # Pad with black frames
    black_frame = np.zeros((224, 224, 3), dtype=np.uint8)

    # E.g. for a stack size of 20, we need a window of 20,so for the last frame, we have the frame itself (+1) and padding (+19) = 20 frames total
    for _ in range(stack_size - 1): # therefore the -1
        out.write(black_frame)

    cap.release()
    out.release()
    print(f"Cropped and padded video saved to: {cropped_path}")

    cropped_cap = cv2.VideoCapture(cropped_path)
    cropped_total_frames = int(cropped_cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cropped_cap.release()
    print(f"Cropped video total frames: {cropped_total_frames} (should be = total frames (of cropped video) + stack size -1)" )


with open(text_file, 'a') as f:
    for root, dirs, files in os.walk(output_folder):
        for file in files:
            if file.endswith(".mp4"):
                full_path = os.path.join(root, file)
                f.write(full_path + '\n')
                print(f"Added: {full_path}")


Match for trial 1: 2025-05-28_001_Freddy-cam-1.mp4
Trial 1 - Start frame index: 0, End frame index: 1135
Total frames in video: 1135
Cropped and padded video saved to: .\sample\2025-05-28_001_Freddy-cam-1_cropped.mp4
Cropped video total frames: 1184 (should be = total frames (of cropped video) + stack size -1)

Match for trial 2: 2025-05-28_002_Freddy-cam-1.mp4
Trial 2 - Start frame index: 0, End frame index: 1068
Total frames in video: 1068
Cropped and padded video saved to: .\sample\2025-05-28_002_Freddy-cam-1_cropped.mp4
Cropped video total frames: 1117 (should be = total frames (of cropped video) + stack size -1)

Match for trial 3: 2025-05-28_003_Freddy-cam-1.mp4
Trial 3 - Start frame index: 8, End frame index: 1093
Total frames in video: 1093
Cropped and padded video saved to: .\sample\2025-05-28_003_Freddy-cam-1_cropped.mp4
Cropped video total frames: 1134 (should be = total frames (of cropped video) + stack size -1)

Match for trial 4: 2025-05-28_004_Freddy-cam-1.mp4
Trial 4 -